# Prompt Essentials Practical Lab

This lab turns prompt engineering into a repeatable workflow: write a baseline prompt, observe the output, improve the prompt contract, template it, and trace it.

You will practice:

- vague prompt vs structured prompt
- zero-shot, one-shot, and few-shot prompting
- sampling settings for reliable vs creative tasks
- system messages as behavior contracts
- reusable LangChain prompt templates
- LangSmith tracing for prompt debugging

This builds on earlier LLM anatomy and agent architecture work. The focus now is communication: how an application tells an LLM what behavior is expected.


## Practical timeline

Use this notebook as a guided classroom lab.

| Lab section | Approx. time | Outcome |
|---|---:|---|
| Setup and environment check | 5 min | Confirm live or fallback mode |
| Bad prompt vs improved prompt | 15 min | See why prompt structure matters |
| Zero-shot / one-shot / few-shot | 20 min | Choose the right example strategy |
| Sampling parameters | 10 min | Match settings to task type |
| System messages and output contracts | 20 min | Control role, boundaries, and format |
| Prompt templates | 15 min | Make prompts reusable |
| LangSmith tracing and debugging lab | 25 min | Inspect and revise prompt failures |


## Setup

The notebook is designed to run in two modes:

- **Live mode:** uses your OpenAI API key.
- **Fallback mode:** uses built-in simulated outputs when the API key is unavailable.

Fallback mode keeps the lesson moving during classroom network issues, but live mode is recommended when possible.


In [ ]:
from pathlib import Path
import csv
import json

from langchain_core.prompts import ChatPromptTemplate

from utils import (
    estimate_demo_cost_usd,
    has_required_keys,
    invoke_messages,
    load_settings,
    make_messages,
    pretty_print,
    print_environment_status,
    print_prompt,
    try_parse_json,
)

settings = load_settings()
print_environment_status(settings)


In [ ]:
# COST NOTE: A full live notebook run is intentionally small.
estimated_cost = estimate_demo_cost_usd(
    approximate_input_tokens=12_000,
    approximate_output_tokens=6_000,
)
print(f"Approximate full-demo cost with gpt-4o-mini pricing assumptions: ${estimated_cost}")


## Load the small teaching datasets

The support-ticket data is used for classification and routing examples.

The travel-request data is used for prompt debugging because travel assistants are easy to make overconfident: they may invent prices, weather certainty, or missing details.


In [ ]:
support_tickets = []
with open("data/support_tickets.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    support_tickets = list(reader)

travel_requests = []
with open("data/travel_requests.jsonl", encoding="utf-8") as f:
    for line in f:
        travel_requests.append(json.loads(line))

print("Support tickets:")
for row in support_tickets[:3]:
    print(f"- {row['ticket_id']}: {row['customer_message']}")

print("\nTravel requests:")
for row in travel_requests:
    print(f"- {row['request_id']}: {row['user_request']}")


# Lab 1 — Bad prompt vs strong prompt

A vague prompt often produces a generic answer. A strong prompt tells the model:

- its role
- the task
- the input
- the desired structure
- what to prioritize
- what to avoid

We will use the same customer complaint twice and compare the outputs.


In [ ]:
customer_complaint = """
I ordered a replacement laptop last week. The support team said it would arrive yesterday,
but the tracking page still says only a shipping label was created. I have a client project
starting tomorrow and nobody has replied to my last two emails.
""".strip()

bad_messages = make_messages(
    system="You are a helpful assistant.",
    user=f"Summarize this customer complaint:\n\n{customer_complaint}",
)

print_prompt(bad_messages)
bad_result = invoke_messages(
    bad_messages,
    temperature=0.2,
    fallback_label="bad_summary",
    run_name="bad_prompt_summary",
)
pretty_print("Bad prompt output", bad_result.output)
if bad_result.error:
    print(f"Live call failed, fallback used: {bad_result.error}")


In [ ]:
strong_messages = make_messages(
    system=(
        "You are a customer support operations analyst. "
        "Your job is to identify the issue, business impact, urgency, and next action. "
        "Do not invent information. If a detail is missing, say it is missing."
    ),
    user=f"""
Analyze the complaint below.

Complaint:
{customer_complaint}

Return exactly four labelled lines:
Issue:
Customer impact:
Urgency:
Recommended action:
""".strip(),
)

print_prompt(strong_messages)
strong_result = invoke_messages(
    strong_messages,
    temperature=0.2,
    fallback_label="better_summary",
    run_name="strong_prompt_summary",
)
pretty_print("Strong prompt output", strong_result.output)
if strong_result.error:
    print(f"Live call failed, fallback used: {strong_result.error}")


### What to notice

The improved prompt does not merely ask for a better answer. It narrows the job.

A useful prompt is closer to a **work order** than a casual question: it specifies what output would count as useful.


# Lab 2 — Zero-shot, one-shot, and few-shot prompting

Use examples only when they teach the model something useful.

- **Zero-shot:** no examples. Good when the task is familiar and low ambiguity.
- **One-shot:** one example. Good when format is the main thing to teach.
- **Few-shot:** multiple examples. Good when labels, edge cases, or style need demonstration.


In [ ]:
ticket = support_tickets[0]["customer_message"]
print(ticket)


In [ ]:
zero_shot_messages = make_messages(
    system="You are a support ticket router.",
    user=f"""
Classify the ticket into one category and one priority.

Allowed categories: Billing, Logistics, Account, Technical
Allowed priorities: Low, Medium, High

Ticket:
{ticket}
""".strip(),
)

zero_result = invoke_messages(
    zero_shot_messages,
    temperature=0.2,
    fallback_label="zero_shot",
    run_name="zero_shot_ticket_router",
)
pretty_print("Zero-shot output", zero_result.output)


In [ ]:
one_shot_messages = make_messages(
    system="You are a support ticket router. Return only valid JSON.",
    user=f"""
Classify the ticket.

Allowed categories: Billing, Logistics, Account, Technical
Allowed priorities: Low, Medium, High

Example:
Ticket: "The replacement laptop has not arrived and the project starts tomorrow."
Output:
{{"category": "Logistics", "priority": "High", "reason": "The delivery delay blocks a time-sensitive project."}}

Now classify this ticket:
Ticket: "{ticket}"
Output:
""".strip(),
)

one_result = invoke_messages(
    one_shot_messages,
    temperature=0.2,
    fallback_label="one_shot",
    run_name="one_shot_ticket_router",
)
pretty_print("One-shot output", one_result.output)


In [ ]:
few_shot_messages = make_messages(
    system="You are a support ticket router. Return only valid JSON with category, priority, and reason.",
    user=f"""
Classify the ticket.

Allowed categories: Billing, Logistics, Account, Technical
Allowed priorities: Low, Medium, High

Example 1:
Ticket: "The replacement laptop has not arrived and the project starts tomorrow."
Output:
{{"category": "Logistics", "priority": "High", "reason": "The delivery delay blocks a time-sensitive project."}}

Example 2:
Ticket: "How do I change my login email?"
Output:
{{"category": "Account", "priority": "Low", "reason": "This is a routine account-change question."}}

Example 3:
Ticket: "The dashboard crashes every time I upload the monthly sales file."
Output:
{{"category": "Technical", "priority": "Medium", "reason": "A product bug blocks a recurring workflow."}}

Now classify this ticket:
Ticket: "{ticket}"
Output:
""".strip(),
)

few_result = invoke_messages(
    few_shot_messages,
    temperature=0.2,
    fallback_label="few_shot",
    run_name="few_shot_ticket_router",
)
pretty_print("Few-shot output", few_result.output)


### Quick comparison

Check which output is easiest to use in an application.

A human can read all three. A program can reliably consume only the outputs with a strong format contract.


In [ ]:
for label, result in [
    ("zero-shot", zero_result),
    ("one-shot", one_result),
    ("few-shot", few_result),
]:
    ok, parsed = try_parse_json(result.output)
    print(f"{label}: JSON parse success = {ok}")
    if ok:
        valid, message = has_required_keys(parsed, {"category", "priority", "reason"})
        print(f"  required keys: {valid} — {message}")
    else:
        print(f"  {parsed}")


# Lab 3 — Sampling parameters in practice

Sampling settings should match the task.

- For routing, extraction, and tool decisions: prefer stable outputs.
- For brainstorming and drafting: allow more variety.
- Avoid changing many parameters at once when debugging.


In [ ]:
draft_prompt = """
Write a short customer update for a delayed laptop replacement.
Keep it professional, empathetic, and under 45 words.
""".strip()

low_temp_messages = make_messages(
    system="You are a concise customer support writer.",
    user=draft_prompt,
)

low_temp_result = invoke_messages(
    low_temp_messages,
    temperature=0.1,
    top_p=0.95,
    fallback_label="temperature_low",
    run_name="low_temperature_draft",
)

high_temp_result = invoke_messages(
    low_temp_messages,
    temperature=0.9,
    top_p=0.95,
    fallback_label="temperature_high",
    run_name="high_temperature_draft",
)

pretty_print("Low temperature output", low_temp_result.output)
pretty_print("High temperature output", high_temp_result.output)


### What to notice

For the same task, higher temperature may produce more expressive wording. That can be useful for writing, but risky for classification or tool-calling.

When building agents, stable behavior usually matters more than charm.


# Lab 4 — System messages as behavior contracts

A system message should shape the assistant's role, boundaries, and output behavior.

We will test a travel-planning assistant. The weak version gives confident advice despite missing details. The stronger version must ask for missing information and avoid inventing live facts.


In [ ]:
travel_request = travel_requests[0]["user_request"]
print(travel_request)


In [ ]:
weak_system_messages = make_messages(
    system="You are a friendly travel assistant.",
    user=travel_request,
)

weak_system_result = invoke_messages(
    weak_system_messages,
    temperature=0.3,
    fallback_label="weak_system",
    run_name="weak_travel_system_message",
)
pretty_print("Weak system-message output", weak_system_result.output)


In [ ]:
strong_system_messages = make_messages(
    system=(
        "You are a careful travel planning assistant. "
        "Do not invent live prices, live weather, availability, or safety guarantees. "
        "When critical information is missing, ask for it instead of pretending to know. "
        "Return only valid JSON with these keys: can_answer, missing_information, safe_next_question, assumptions_made."
    ),
    user=travel_request,
)

strong_system_result = invoke_messages(
    strong_system_messages,
    temperature=0.2,
    fallback_label="strong_system",
    run_name="strong_travel_system_message",
)
pretty_print("Strong system-message output", strong_system_result.output)


In [ ]:
ok, parsed = try_parse_json(strong_system_result.output)
print(f"JSON parse success: {ok}")

if ok:
    valid, message = has_required_keys(
        parsed,
        {"can_answer", "missing_information", "safe_next_question", "assumptions_made"},
    )
    print(f"Required key check: {valid} — {message}")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
else:
    print(parsed)


### What to notice

The system message does three jobs:

1. Defines the role.
2. Sets boundaries.
3. Makes the output checkable.

This is the habit that later becomes important in agents, RAG systems, and guardrails.


# Lab 5 — Prompt templates with LangChain

Hardcoded prompts are fine for the first experiment. They become fragile when the application grows.

A prompt template lets you keep the prompt contract stable while changing the input variables.


In [ ]:
ticket_router_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a support operations analyst. "
        "Return only valid JSON with category, priority, customer_impact, and next_action. "
        "Allowed categories: Billing, Logistics, Account, Technical. "
        "Allowed priorities: Low, Medium, High. "
        "Do not invent facts that are not in the ticket."
    ),
    (
        "human",
        "Ticket ID: {ticket_id}\nCustomer message: {customer_message}"
    ),
])

selected_ticket = support_tickets[1]
templated_messages = ticket_router_template.format_messages(
    ticket_id=selected_ticket["ticket_id"],
    customer_message=selected_ticket["customer_message"],
)

print_prompt(templated_messages)

template_result = invoke_messages(
    templated_messages,
    temperature=0.2,
    fallback_label="template_ticket",
    run_name="templated_ticket_router",
)
pretty_print("Templated prompt output", template_result.output)


In [ ]:
ok, parsed = try_parse_json(template_result.output)
print(f"JSON parse success: {ok}")

if ok:
    valid, message = has_required_keys(
        parsed,
        {"category", "priority", "customer_impact", "next_action"},
    )
    print(f"Required key check: {valid} — {message}")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
else:
    print(parsed)


### Template habit

A good prompt template is a reusable **contract**:

- stable instructions
- variable inputs
- expected output format
- clear constraints

This is the bridge from manual prompting to application development.


# Lab 6 — LangSmith tracing

LangSmith is useful because prompt debugging needs visibility.

When tracing is enabled, each live model call can be inspected: input messages, model output, latency, metadata, and errors.

To use tracing:

1. Create a LangSmith account.
2. Add `LANGSMITH_TRACING=true` to `.env`.
3. Add `LANGSMITH_API_KEY`.
4. Add `LANGSMITH_PROJECT=BIA_Prompt_Essentials`.
5. Restart the kernel and rerun a model call.

The next cell prints whether tracing appears enabled in this environment.


In [ ]:
settings = load_settings()
print_environment_status(settings)

print("\nTrainer action:")
print("If tracing is enabled, open LangSmith and look for runs named like:")
print("- bad_prompt_summary")
print("- templated_ticket_router")
print("- strong_travel_system_message")


# Lab 7 — Prompt debugging lab

Now we use a deliberately flawed prompt.

The user asks for a travel plan with missing constraints and a weather guarantee. A poor prompt will often produce confident, unsafe, or unverifiable claims.

Your job is to improve the prompt so that it:

- does not invent live weather or prices
- asks for missing details
- returns a checkable JSON object
- explains risk flags clearly


In [ ]:
debug_request = travel_requests[0]["user_request"]

debug_bad_messages = make_messages(
    system="You are an expert travel planner. Give the user a useful answer.",
    user=debug_request,
)

debug_bad_result = invoke_messages(
    debug_bad_messages,
    temperature=0.4,
    fallback_label="debug_bad",
    run_name="debug_lab_bad_prompt",
)
pretty_print("Flawed prompt output", debug_bad_result.output)


### Diagnose the failure

Before running the improved prompt, write down the likely failure mode:

- Did the answer invent facts?
- Did it ignore missing dates or budget?
- Did it promise certainty?
- Could another program parse the answer?


In [ ]:
debug_better_messages = make_messages(
    system=(
        "You are a careful travel planning assistant. "
        "You must not invent live weather, prices, hotel availability, or crowd levels. "
        "If the request contains missing information or impossible guarantees, ask clarifying questions. "
        "Return only valid JSON with these keys: answer_status, risk_flags, clarifying_questions, safe_response."
    ),
    user=f"""
User request:
{debug_request}

Decide whether you can answer safely. If not, ask for the minimum missing details.
""".strip(),
)

debug_better_result = invoke_messages(
    debug_better_messages,
    temperature=0.2,
    fallback_label="debug_better",
    run_name="debug_lab_better_prompt",
)
pretty_print("Improved prompt output", debug_better_result.output)


In [ ]:
ok, parsed = try_parse_json(debug_better_result.output)
print(f"JSON parse success: {ok}")

if ok:
    valid, message = has_required_keys(
        parsed,
        {"answer_status", "risk_flags", "clarifying_questions", "safe_response"},
    )
    print(f"Required key check: {valid} — {message}")
    print(json.dumps(parsed, indent=2, ensure_ascii=False))
else:
    print(parsed)


# Integration mini-system

The final cell combines the core habits:

1. Load an input.
2. Use a template.
3. Apply a strong system message.
4. Run at a stable temperature.
5. Validate the output.
6. Inspect the trace if LangSmith is enabled.


In [ ]:
travel_safety_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a careful travel intake assistant. "
        "Your job is not to produce the full itinerary yet. "
        "Your job is to check whether enough information exists to plan safely. "
        "Never invent live prices, weather, visa rules, availability, or safety guarantees. "
        "Return only valid JSON with prompt_version, result_quality, format_valid, and trace_hint."
    ),
    (
        "human",
        "User travel request: {request}\nTeaching point: {teaching_point}"
    ),
])

selected_request = travel_requests[2]
integration_messages = travel_safety_template.format_messages(
    request=selected_request["user_request"],
    teaching_point=selected_request["teaching_point"],
)

integration_result = invoke_messages(
    integration_messages,
    temperature=0.2,
    fallback_label="integration",
    run_name="integration_prompt_debugging_workflow",
)

pretty_print("Integration output", integration_result.output)

ok, parsed = try_parse_json(integration_result.output)
print(f"JSON parse success: {ok}")
if ok:
    print(json.dumps(parsed, indent=2, ensure_ascii=False))


# Exercises

Try these after the guided lab.

## Exercise 1 — Add an edge case

Add a ticket where the customer reports both a billing issue and a technical issue. Decide which category should win and add a few-shot example that teaches the priority rule.

Hint: add the rule to the system message before adding more examples.

## Exercise 2 — Rewrite the travel prompt

Create a system message for a travel assistant that must ask for missing dates, budget, travellers, and departure city before planning.

Hint: require JSON with `can_plan`, `missing_fields`, and `next_question`.

## Exercise 3 — Trace comparison

Run the weak and strong travel prompts with LangSmith tracing enabled. Compare the input prompt, output, and token usage.

Hint: give each call a clear `run_name`.


# Wrap-up

You have practiced the workflow that matters most:

- design the prompt contract
- test the output
- compare prompt versions
- validate structure
- inspect traces
- improve the prompt based on a specific failure

The next step is structured prompting, where the prompt does more than answer once: it can guide reasoning, tool use, and revision loops.
